In [1]:
import pymc as pm
import pandas as pd
import numpy as np

In [2]:
import importlib

# 1. FORCE JUPYTER TO RELOAD THE UPDATED 2D MODEL
import rosenbrock
importlib.reload(rosenbrock)
from rosenbrock import make_model

In [3]:
data = {
  "n1": 2,
  "n2": 1,
  "mu": 1.0,
  "a": 0.05,
  "b": 5.0
}
pymc_model = make_model(data)


In [4]:
file_path = "Rdraws/"
stan_draws = pd.read_csv(file_path + "draws_rosenbrock.csv")
stan_draws = stan_draws.drop(columns=["lp__"])
stan_eval_points = pd.read_csv(file_path + "eval_points_rosenbrock.csv")
stan_grads = pd.read_csv(file_path + "grads_rosenbrock.csv")
stan_log_probs = pd.read_csv(file_path + "log_probs_rosenbrock.csv")

In [5]:
logp_func = pymc_model.compile_logp()
dlogp_func = pymc_model.compile_dlogp()

# 3. Clean Stan Data
stan_eval = stan_eval_points.drop(columns=["lp__"], errors="ignore")
stan_g = stan_grads.drop(columns=["lp__"], errors="ignore").values
# Explicitly find the columns by name so ordering doesn't break the math
x1_col = "x1"
x_rest_cols = [c for c in stan_eval.columns if "x_rest" in c]

pymc_log_probs = []
pymc_grads = []

In [6]:
# 4. Evaluate the PyMC model
for _, row in stan_eval.iterrows():
    point_dict = {
        # Explicitly cast to numpy arrays to keep PyTensor's JIT backend happy
        "x1": np.array(row[x1_col], dtype=np.float64),
        "x_rest": row[x_rest_cols].values.reshape((data["n2"], data["n1"] - 1), order='F').astype(np.float64)
    }
    logp = logp_func(point_dict)
    pymc_log_probs.append(logp)
    
    
    grad_list = dlogp_func(point_dict)
    # Flatten the PyMC gradient list [grad_x1, grad_x_rest] into a single vector
    flat_grad = np.concatenate([np.atleast_1d(g).flatten(order='F') for g in grad_list])
    pymc_grads.append(flat_grad)

pymc_log_probs = np.array(pymc_log_probs)
pymc_grads = np.array(pymc_grads)
stan_lp = stan_log_probs.values.flatten()

In [7]:
# 5. Check differences
logp_diff = np.max(np.abs(pymc_log_probs - stan_lp))
grad_diff = np.max(np.abs(pymc_grads - stan_g))

print(f"Max absolute log-prob difference: {logp_diff:.2e}")
print(f"Max absolute gradient difference: {grad_diff:.2e}")

np.testing.assert_allclose(pymc_log_probs, stan_lp, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(pymc_grads, stan_g, atol=1e-5, rtol=1e-5)
print("\nSuccess: PyMC and Stan implementations are mathematically identical!")

Max absolute log-prob difference: 2.45e-12
Max absolute gradient difference: 1.98e-10

Success: PyMC and Stan implementations are mathematically identical!
